# Stage 8B-3B — Multi-\(W\), Multi-\(z\) Symmetric Gamma–Gamma Label Engine

B3-A sonucundan:

\[
\boxed{N_{\rm MC}=64000}
\]

kilitlendi.

Şimdi bir bank/environment için stochastic kanalı **her candidate için yeniden üretmeyeceğiz**.

Bir MC chunk'ta:

\[
H_{BR}^{(n)},H_{RU}^{(n)}
\]

yalnızca **bir kere** üretilir ve bütün

\[
K\ W \times C\ z
\]

candidate'ları tarafından paylaşılır.

Final label:

\[
\boxed{
q_{05,GG}(x,W_k,z_c)
=
GG_{0.05}
\left(
\mu_{\rm SNR}^{\rm analytic}(x,W_k,z_c),
\operatorname{varEmp}^{MC}(x,W_k,z_c)
\right)
}
\]

Raw empirical percentile kullanılmaz.

Bu notebook üç şeyi yapar:

1. Yeni batched contraction'ı eski `F -> Feff -> Y` yolu ile küçük ölçekte doğrular.
2. Hızlı analitik mean kernel'ını Stage-3 full analitik yol ile doğrular.
3. \(K=32,\ C=512,\ N=64000\) production benchmark çalıştırır ve **GG labels/s** ölçer.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys, shutil, time
import numpy as np
import pandas as pd
import torch

ROOT=Path('/content/drive/MyDrive/MyDrive/RIS')

required=[
    'ris_gpu_physics_stage1.py',
    'ris_gpu_rho_stage2.py',
    'ris_gpu_stats_stage3.py',
    'ris_gpu_geometry_lsp_stage67.py',
    'ris_gpu_environment_stage8.py',
    'ris_gpu_channel_realizations_stage8b1.py',
    'ris_gpu_channel_native_stage8b2.py',
    'ris_gpu_precoder_stage5.py',
    'ris_gpu_ris_response_stage4.py',
    'stage8b2_dataset_mean_var_validation.py',
    'ris_gpu_symmetric_gg_stage8b3.py',
    'ris_gpu_symmetric_gg_labels_stage8b3b.py',
]

for d in [ROOT,Path('/content')]:
    if str(d) not in sys.path:
        sys.path.insert(0,str(d))

missing=[
    f for f in required
    if not (ROOT/f).exists() and not (Path('/content')/f).exists()
]
assert not missing, "Eksik:\n"+"\n".join(missing)

from ris_gpu_symmetric_gg_stage8b3 import build_gg_lookup_from_dataset
from ris_gpu_symmetric_gg_labels_stage8b3b import (
    row_to_bank_input,
    build_w_candidate_pool,
    build_z_candidate_pool,
    estimate_candidate_chunk_memory_mb,
    validate_batched_contraction,
    validate_mean_only_against_stage3,
    run_symmetric_gg_label_engine,
    flatten_label_result,
)
from ris_gpu_environment_stage8 import build_deterministic_bank

device='cuda' if torch.cuda.is_available() else 'cpu'
print("Device:",device)
if torch.cuda.is_available():
    print("GPU:",torch.cuda.get_device_name(0))

In [ ]:
DRIVE_CSV=ROOT/'variance_training_dataset_corrected_mu_rho_cf_v3.csv'
assert DRIVE_CSV.exists(),DRIVE_CSV

LOCAL_CSV=Path('/content/variance_training_dataset_corrected_mu_rho_cf_v3.csv')

if (
    not LOCAL_CSV.exists()
    or LOCAL_CSV.stat().st_size != DRIVE_CSV.stat().st_size
):
    print("CSV local SSD'ye kopyalanıyor...")
    shutil.copy2(DRIVE_CSV,LOCAL_CSV)

D=pd.read_csv(LOCAL_CSV)

print("Rows:",len(D))
print("nRIS:",sorted(D.nRIS.unique()))

lookup=build_gg_lookup_from_dataset(D)
print("GG lookup points:",len(lookup.log_cv2))

## Benchmark bank

İlk production benchmark'ı ağır ama aşırı uç olmayan:

\[
nRIS=512,\quad nT\ge16,\quad nR\le8
\]

bir `test_interpolation` bankında yapıyoruz.

Böylece \(32W\times512z\) gerçekçi biçimde test edilir.

In [ ]:
cand=D[
    (D.splitID.astype(str)=='test_interpolation')
    & (D.nRIS.astype(int)==512)
    & (D.nT.astype(int)>=16)
    & (D.nR.astype(int)<=8)
].sort_values(['bankID','pairID'])

assert len(cand)>0

row=cand.iloc[0]

print(
    "Bank:",
    int(row.bankID),
    "|",row.scenario_BR,"/",row.scenario_RU,
    "| nT=",int(row.nT),
    "nR=",int(row.nR),
    "nRIS=",int(row.nRIS),
)

## 1. Batched empirical contraction correctness

Küçük bir testte:

\[
N=32,\quad K=3,\quad C=5
\]

için yeni GEMM motoru ile eski doğrulanmış:

\[
H_{RU}\operatorname{diag}(\gamma)H_{BR}W
\]

yolu birebir karşılaştırılır.

In [ ]:
check=validate_batched_contraction(
    row,
    n_mc=32,
    k_w=3,
    c_z=5,
    device=device,
    parity=False,
)

print(check)

assert check['relative_fro'] < 2e-5
assert check['max_abs'] < 2e-3

print("PASS — batched multi-W/multi-z empirical contraction")

## 2. Deterministic analytical environment'i bir kere hazırla

B3-B empirical kanalı ve analitik mean'i ayrı kullanır.

\[
\rho,\ \mu_H,\ \sigma_H^2
\]

gibi environment state bank başına bir kere hazırlanır.

In [ ]:
bank_input=row_to_bank_input(row)

if torch.cuda.is_available():
    torch.cuda.synchronize()
t0=time.perf_counter()

bank_state=build_deterministic_bank(
    bank_input,
    device=device,
    parity=False,
)

if torch.cuda.is_available():
    torch.cuda.synchronize()

print(
    "Deterministic bank prepare:",
    f"{time.perf_counter()-t0:.3f}s"
)
print(bank_state['timings'])

static_env=bank_state['static_env']

## 3. \(32W\) ve \(512z\) candidate pool

In [ ]:
K_W=32
C_Z=512

W,WIdx=build_w_candidate_pool(
    row,k_w=K_W,device=device,parity=False
)

z,gamma=build_z_candidate_pool(
    int(row.nRIS),
    c_z=C_Z,
    seed=20260818,
    device=device,
    parity=False,
)

print("W:",tuple(W.shape))
print("WIdx:",tuple(WIdx.shape))
print("z:",tuple(z.shape))
print("gamma:",tuple(gamma.shape))
print("Total candidate labels:",K_W*C_Z)

## 4. Fast analytic mean correctness

Yeni B3-B mean yolu sadece \(UBR\) ve effective second moment kernel'ını üretir;
`Cmat` ve `sigma2Wick` üretmez.

Küçük bir altkümede Stage-3 full analitik fonksiyonuyla karşılaştırılır.

In [ ]:
mean_check=validate_mean_only_against_stage3(
    static_env,
    W,
    gamma,
    k_check=2,
    c_check=4,
)

print(mean_check)

assert mean_check['relative_fro'] < 2e-5
assert mean_check['max_abs'] < 2e-3

print("PASS — lightweight analytic muSNR")

## 5. Memory/chunk ayarı

İlk güvenli production ayarı:

\[
N_{\rm chunk}=256,\quad
K_{\rm chunk}=4,\quad
C_{\rm chunk}=64.
\]

Bu ayarlar toplam \(N=64000\) değerini değiştirmez; sadece GPU memory/throughput
dengesini belirler.

In [ ]:
N_MC=64_000
MC_CHUNK=256
W_CHUNK=4
Z_CHUNK=64

mem=estimate_candidate_chunk_memory_mb(
    mc_chunk=MC_CHUNK,
    w_chunk=W_CHUNK,
    z_chunk=Z_CHUNK,
    n_ris=int(row.nRIS),
    n_r=int(row.nR),
    n_t=int(row.nT),
)

display(pd.DataFrame([mem]))

print(
    "Not: bu yaklaşık değer stochastic generator'ın kendi geçici "
    "primitive tensorlerini içermez."
)

# 6. FULL 8B3-B benchmark

Bu hücre:

\[
\boxed{
64000 \times 32 \times 512
}
\]

candidate-sample evaluation yapar.

Fakat channel generation sayısı sadece:

\[
\boxed{64000\ BR/RU\ realization\ pair}
\]

olur.

Ana performans çıktıları:

- empirical contraction süresi,
- analytic mean süresi,
- GG lookup süresi,
- total süre,
- sample-evaluations/s,
- **final Symmetric-GG labels/s**,
- peak CUDA memory.

In [ ]:
if torch.cuda.is_available():
    torch.cuda.synchronize()

T0=time.perf_counter()

result=run_symmetric_gg_label_engine(
    row,
    static_env,
    W,
    gamma,
    lookup,
    n_mc=N_MC,
    mc_chunk=MC_CHUNK,
    w_chunk=W_CHUNK,
    z_chunk=Z_CHUNK,
    device=device,
    parity=False,
)

if torch.cuda.is_available():
    torch.cuda.synchronize()

wall=time.perf_counter()-T0

print("\n"+"="*86)
print("STAGE 8B-3B — MULTI-W / MULTI-z SYMMETRIC GG")
print("="*86)

print(f"Bank                       : {int(row.bankID)}")
print(f"nT / nR / nRIS             : {int(row.nT)} / {int(row.nR)} / {int(row.nRIS)}")
print(f"N_MC                       : {N_MC:,}")
print(f"W candidates               : {K_W}")
print(f"z candidates               : {C_Z}")
print(f"Final labels               : {result['candidate_count']:,}")
print()
print(f"Empirical MC + contraction : {result['empirical_seconds']:.3f} s")
print(f"Analytic muSNR             : {result['analytic_mu_seconds']:.3f} s")
print(f"Symmetric-GG lookup        : {result['gg_lookup_seconds']:.6f} s")
print(f"Engine total               : {result['total_seconds']:.3f} s")
print(f"Notebook wall              : {wall:.3f} s")
print()
print(
    "Sample evaluations/s       :",
    f"{result['sample_evaluations_per_second']:,.0f}"
)
print(
    "Empirical candidate/s      :",
    f"{result['empirical_candidate_labels_per_second']:,.2f}"
)
print(
    "FINAL GG LABELS/s          :",
    f"{result['labels_per_second']:,.2f}"
)
print(
    "Peak CUDA allocated        :",
    f"{result['peak_memory_MB']:.1f} MB"
)
print(
    "Lookup clamped             :",
    f"{100*result['lookupClamped'].double().mean().item():.3f}%"
)

## 7. Sanity checks

Analitik mean ile 64k empirical mean aynı distribution'ın mean'ini temsil
ettiği için yakın olmalıdır.

Bu, B2'de yaptığımız doğrulamanın multi-candidate versiyonudur.

In [ ]:
mu=result['muSNR'].detach().cpu().numpy()
mean_emp=result['meanEmp'].detach().cpu().numpy()
var_emp=result['varEmp'].detach().cpu().numpy()
q05=result['q05GG'].detach().cpu().numpy()

mean_rel=np.abs(mean_emp-mu)/np.maximum(np.abs(mu),np.finfo(float).eps)

print("Analytic mean vs 64k empirical mean")
print("  MdAPE :",f"{100*np.median(mean_rel):.3f}%")
print("  P90   :",f"{100*np.quantile(mean_rel,.90):.3f}%")
print("  P95   :",f"{100*np.quantile(mean_rel,.95):.3f}%")
print("  Max   :",f"{100*np.max(mean_rel):.3f}%")

assert np.isfinite(var_emp).all()
assert (var_emp>0).all()
assert np.isfinite(q05).all()
assert (q05>=0).all()

print()
print("q05GG:")
print("  min   :",np.min(q05))
print("  median:",np.median(q05))
print("  max   :",np.max(q05))

## 8. Candidate dataset çıktısı

Bu dosya artık **bir bank için 16,384 gerçek B3-B label** içerir:

\[
32W\times512z.
\]

Her satırda:

- `WIdx`
- `zString`
- analytic `muSNR`
- MC `meanEmp`
- MC `varEmp`
- symmetric-GG `shapeA`
- final `q05GG`

bulunur.

In [ ]:
labels=flatten_label_result(
    result,
    WIdx,
    z,
)

labels.insert(0,'bankID',int(row.bankID))
labels.insert(1,'scenario_BR',str(row.scenario_BR))
labels.insert(2,'scenario_RU',str(row.scenario_RU))
labels.insert(3,'nRIS',int(row.nRIS))
labels.insert(4,'nT',int(row.nT))
labels.insert(5,'nR',int(row.nR))
labels.insert(6,'N_MC',N_MC)

OUT=Path('/content/stage8b3b_example_bank_labels.parquet')
labels.to_parquet(OUT,index=False,compression='zstd')

print("Saved:",OUT)
print("Rows :",len(labels))
display(labels.head())

# B3-B PASS sonrası

Bu benchmark başarılıysa Stage 8B artık kapanır.

Sonraki adım **Stage 8C production GitHub refactor**:

```text
src/ris_env/
    channel_realizations.py
    effective_moments.py
    snr_statistics.py
    gamma_gamma.py
    environment.py
```

ve ardından yeni Symmetric-GG q05 dataset generator'ı hazırlanır.

B3-B'deki bu kernel final repo'da:

```python
bank.symmetric_gg_q05_batch(
    W=W_batch,
    z=z_batch,
    n_realizations=64000,
)
```

API'sinin temelini oluşturur.